![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/36.2.MedicalLLMEntityExtractor.ipynb)

If you are using the `spark-nlp-jsl` library, please use this  [46.2.MedicalLLMEntityExtractor](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/46.2.MedicalLLMEntityExtractor.ipynb) notebook.

# MedicalLLMEntityExtractor

## Overview

`MedicalLLMEntityExtractor` is an end-to-end LLM-based named entity recognition (NER) annotator for clinical and healthcare text. It leverages Large Language Models (LLMs) in GGUF format with **structured JSON output enforced via BNF grammars**, ensuring reliable and parseable entity extraction from any medical document.

### How It Works

The annotator follows the **LangExtract** pattern from Google Research, combining three components:

1. **Few-shot prompting** — optional `(input, json_output)` example pairs that guide the LLM toward the desired extraction format
2. **BNF grammar-constrained generation** — llama.cpp enforces valid JSON output at every generated token, preventing malformed responses
3. **String matching** — character-level `begin`/`end` offsets are computed by locating extracted spans in the original source text

### Annotator Output Format

The annotator then converts this JSON into Spark NLP **CHUNK** annotations with accurate character offsets resolved via string matching:

```bash
{chunk, 46,  64,  shortness of breath, {entity -> PROBLEM,   chunk -> 0, sentence -> 0, ner_source -> entities}, []}
{chunk, 125, 127, BNP,                 {entity -> TEST,      chunk -> 1, sentence -> 0, ner_source -> entities}, []}
{chunk, 319, 328, furosemide,          {entity -> TREATMENT, chunk -> 2, sentence -> 0, ner_source -> entities}, []}
```

Each CHUNK annotation field:

| Field | Description |
|---|---|
| `annotatorType` | Always `chunk` |
| `begin` | Start character offset in the source text |
| `end` | End character offset in the source text |
| `result` | The exact extracted text span (matched from source) |
| `metadata["entity"]` | Entity type label (e.g., `PROBLEM`, `DRUG`, `DATE`) |
| `metadata["chunk"]` | Chunk index within the document |
| `metadata["sentence"]` | Source sentence index from `SentenceDetectorDLModel` |
| `metadata["ner_source"]` | The output column name of the extractor |

### Input / Output Annotation Types

| Input Annotation | Output Annotation |
|---|---|
| `DOCUMENT` | `CHUNK` |

### Key Features

| Feature | Description |
|---|---|
| Zero-shot NER | Works out-of-the-box with just entity type names — no examples required |
| Custom entity types | Define any clinical or domain-specific entity types at inference time |
| Type descriptions | Use `::` syntax: `"DRUG::The exact drug name as it appears in text"` |
| Few-shot examples | Provide `(text, json)` tuples to boost extraction accuracy |
| Custom prompt | Override the full prompt template with `setPromptTemplate()` |
| Accurate offsets | Character-level positions via string matching, not LLM generation |
| Grammar-enforced JSON | BNF grammars prevent malformed JSON output |

### MedicalLLMEntityExtractor vs MedicalNerModel

| | `MedicalLLMEntityExtractor` | `MedicalNerModel` |
|---|---|---|
| Architecture | LLM (GGUF) | BiLSTM-CRF |
| Entity types | Defined at runtime — no retraining needed | Fixed at training time |
| New entity types | Instantly, via `setEntityTypes()` | Requires fine-tuning |
| Few-shot learning | Built-in via `setFewShotExamples()` | N/A |
| Output type | `CHUNK` (with offsets) | `NAMED_ENTITY` → `CHUNK` via converter |
| Speed | Slower (LLM inference) | Fast (neural sequence labeling) |
| Best for | New domains, flexible types, zero-shot | High-throughput, fixed schemas |

## Setup

In [ ]:
# Install the johnsnowlabs library to access Spark NLP for Healthcare
! pip install -q johnsnowlabs

In [ ]:
from google.colab import files

print('Please upload your John Snow Labs license file using the button below')
license_keys = files.upload()

In [ ]:
from johnsnowlabs import nlp, medical

# Install all licensed Python wheels and pre-download JVM jars
nlp.settings.enforce_versions = False
nlp.install(refresh_install=True)

In [ ]:
from johnsnowlabs import nlp, medical
from pyspark.sql import functions as F

import warnings
warnings.filterwarnings('ignore')

spark_conf = {
    "spark.driver.memory": "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
}

# Automatically load license data and start a session with all jars
spark = nlp.start(
    spark_conf=spark_conf,
    # hardware_target="gpu"  # Uncomment for GPU acceleration
)

print("Spark NLP Version     :", nlp.version())
print("Spark NLP JSL Version :", medical.version())
spark

## Supported NER Models

The following JSL models can be used with `MedicalLLMEntityExtractor`.

| Model Name | Disk Size | Model Size | Available Quantizations | GPU Memory | Token/sec | Tasks |
|---|---|---|---|---|---|---|
| JSL_MedS_NER_v4 | 2.2G | 3.5B | [q4](https://nlp.johnsnowlabs.com/2025/07/01/jsl_meds_ner_q4_v4_en.html), [q8](https://nlp.johnsnowlabs.com/2025/07/01/jsl_meds_ner_q8_v4_en.html) | 10GB | 28.5 | Extract and link medical named entities |
| JSL_MedS_4B_v5 | 5.7G | 4B | [q4](https://nlp.johnsnowlabs.com/2025/08/05/jsl_meds_4b_q4_v5_en.html), [q16](https://nlp.johnsnowlabs.com/2025/08/05/jsl_meds_4b_q16_v5_en.html) | 16GB | 83 | NER, Q&A, Summarization, RAG |
| JSL_MedS_8B_v4 | 7.8G | 8B | [q4](https://nlp.johnsnowlabs.com/2025/08/05/jsl_meds_8b_q4_v4_en.html), [q8](https://nlp.johnsnowlabs.com/2025/08/05/jsl_meds_8b_q8_v4_en.html), [q16](https://nlp.johnsnowlabs.com/2025/08/05/jsl_meds_8b_q16_v4_en.html) | 16GB | 84 | NER, Q&A, Summarization, RAG |
| JSL_MedM_v3 | 14G | 14B | [q4](https://nlp.johnsnowlabs.com/2024/10/06/jsl_medm_q4_v3_en.html), [q8](https://nlp.johnsnowlabs.com/2024/10/08/jsl_medm_q8_v3_en.html) | 24GB | 79 | NER, Q&A, Summarization, RAG, Chat |

> **Tip**: `jsl_meds_4b_q16_v5` is the default model used in this notebook — a 4B general-purpose clinical LLM that delivers strong NER accuracy alongside Q&A, summarization, and RAG capabilities.

## Load Model

Load `SentenceDetectorDLModel` and `MedicalLLMEntityExtractor` once and reuse them across all examples. Each document is automatically split into sentences before extraction — this handles multi-sentence documents of any length and keeps the LLM context usage per inference manageable.

In [ ]:
document_assembler = nlp.DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence_detector = nlp.SentenceDetectorDLModel.pretrained("sentence_detector_dl_healthcare", "en", "clinical/models") \
    .setInputCols(["document"]) \
    .setOutputCol("sentence")

entity_extractor = medical.MedicalLLMEntityExtractor.pretrained("jsl_meds_4b_q16_v5", "en", "clinical/models") \
    .setInputCols(["sentence"]) \
    .setOutputCol("entities") \
    .setNGpuLayers(99) \
    .setNCtx(4096) \
    .setNPredict(500) \
    .setTemperature(0.1) \
    .setBatchSize(4) \
    .setTopK(40) \
    .setTopP(0.9)

# 1. Default Clinical Entity Extraction — PROBLEM / TEST / TREATMENT

The default entity types for `MedicalLLMEntityExtractor` are `PROBLEM`, `TEST`, and `TREATMENT` — the three core clinical NER categories aligned with the i2b2 2010 challenge schema.

- **PROBLEM** — any medical condition, disease, symptom, or clinical finding
- **TEST** — diagnostic tests, laboratory investigations, imaging studies, or clinical procedures used to assess a condition
- **TREATMENT** — medications, surgical procedures, therapeutic interventions, or management plans

This example processes inpatient discharge summaries covering cardiology, rheumatology, infectious disease, and neurology.

In [ ]:
entity_extractor \
    .setEntityTypes([
        "PROBLEM::A medical condition, disease, symptom, or clinical finding present in the patient.",
        "TEST::A diagnostic test, lab investigation, imaging study, or procedure used to assess a condition.",
        "TREATMENT::A medication, drug, surgical procedure, or therapeutic intervention.",
    ]) \
    .setFewShotExamples([]) \
    .setPromptTemplate("")

pipeline_1 = nlp.Pipeline(stages=[document_assembler, sentence_detector, entity_extractor])

In [ ]:
discharge_summaries = [
    "A 58-year-old male presented with progressive shortness of breath and bilateral lower extremity "
    "edema over the past 2 weeks. BNP was markedly elevated at 1,200 pg/mL. Echocardiogram demonstrated "
    "severely reduced ejection fraction of 25%. Patient was diagnosed with acute decompensated heart failure "
    "and initiated on IV furosemide 40mg, carvedilol 6.25mg BID, and lisinopril 5mg daily.",

    "45-year-old female with a 10-year history of rheumatoid arthritis presenting for medication management. "
    "ESR 88 mm/hr and CRP 3.4 mg/dL both significantly elevated. X-rays of bilateral hands show progressive "
    "joint erosion and narrowing. Methotrexate 15mg weekly has proven insufficient. Plan: add adalimumab "
    "40mg SQ every 2 weeks and continue hydroxychloroquine 200mg BID.",

    "65-year-old male with type 2 diabetes mellitus admitted with fever, chills, and right flank pain "
    "for 3 days. Urinalysis: positive for nitrites, leukocyte esterase, and WBCs. Urine culture pending. "
    "CT abdomen and pelvis revealed right hydronephrosis with perinephric fat stranding consistent with "
    "acute pyelonephritis. Initiated IV ceftriaxone 1g Q24H with plan to step down to oral ciprofloxacin "
    "500mg BID upon clinical improvement.",

    "Patient is a 72-year-old woman with a history of atrial fibrillation, hypertension, and chronic "
    "kidney disease Stage 3 presenting with acute ischemic stroke. MRI brain shows left MCA territory "
    "infarction. INR on admission 1.1. Initiated IV alteplase thrombolysis within 3 hours of symptom onset. "
    "Anticoagulation with apixaban 5mg BID started after 24-hour CT follow-up. Aspirin and clopidogrel held.",
]

data_1 = spark.createDataFrame(
    [(f"doc_{i+1}", t) for i, t in enumerate(discharge_summaries)]
).toDF("doc_id", "text")
data_1.show(20, truncate=100)

In [ ]:
%%time
result_1 = pipeline_1.fit(data_1).transform(data_1).cache()
result_1.select("doc_id", "entities.result").show(20, truncate=100)

In [ ]:
result_1.selectExpr("doc_id", "explode(entities) AS entity") \
    .select(
        F.col("doc_id"),
        F.col("entity.metadata")["sentence"].cast("int").alias("sentence_id"),
        F.col("entity.result").alias("entity_text"),
        F.col("entity.begin").alias("begin"),
        F.col("entity.end").alias("end"),
        F.col("entity.metadata")["entity"].alias("entity_type"),
    ).show(20, truncate=100)

# 2. Medication & Prescription Information Extraction

Medication extraction is one of the most clinically impactful NER tasks. This example demonstrates extraction of five medication-related entity types from discharge medication lists and ICU order sets:

- **DRUG** — the medication or drug name
- **DOSAGE** — the dose amount and unit (e.g., `500mg`, `1g`, `75 mcg/kg/min`)
- **ROUTE** — the administration route (e.g., `PO`, `IV`, `SQ`, `inhaled`)
- **FREQUENCY** — the dosing schedule (e.g., `BID`, `Q12H`, `daily`, `PRN`)
- **DURATION** — the treatment duration (e.g., `7 days`, `6 weeks`)

Few-shot examples are provided to improve accuracy on clinical abbreviations (BID, TID, Q6H, etc.).

In [ ]:
medication_few_shot = [
    (
        "Patient prescribed amoxicillin 500mg PO TID for 10 days.",
        '{"extractions": [{"entity": "DRUG", "text": "amoxicillin"}, '
        '{"entity": "DOSAGE", "text": "500mg"}, '
        '{"entity": "ROUTE", "text": "PO"}, '
        '{"entity": "FREQUENCY", "text": "TID"}, '
        '{"entity": "DURATION", "text": "10 days"}]}'
    ),
    (
        "Vancomycin 1.25g IV Q12H x 7 days for MRSA bacteremia.",
        '{"extractions": [{"entity": "DRUG", "text": "Vancomycin"}, '
        '{"entity": "DOSAGE", "text": "1.25g"}, '
        '{"entity": "ROUTE", "text": "IV"}, '
        '{"entity": "FREQUENCY", "text": "Q12H"}, '
        '{"entity": "DURATION", "text": "7 days"}]}'
    ),
    (
        "Metformin 1000mg PO twice daily with meals and lisinopril 10mg PO once daily.",
        '{"extractions": [{"entity": "DRUG", "text": "Metformin"}, '
        '{"entity": "DOSAGE", "text": "1000mg"}, '
        '{"entity": "ROUTE", "text": "PO"}, '
        '{"entity": "FREQUENCY", "text": "twice daily"}, '
        '{"entity": "DRUG", "text": "lisinopril"}, '
        '{"entity": "DOSAGE", "text": "10mg"}, '
        '{"entity": "ROUTE", "text": "PO"}, '
        '{"entity": "FREQUENCY", "text": "once daily"}]}'
    ),
]

entity_extractor \
    .setEntityTypes([
        "DRUG::The exact medication or drug name as written in the text. Examples: aspirin, metformin, vancomycin.",
        "DOSAGE::The exact dose amount including units. Examples: 500mg, 1.25g, 40mg, 0.1 mcg/kg/min.",
        "ROUTE::The administration route. Examples: PO, IV, SQ, IM, inhaled, sublingual, topical.",
        "FREQUENCY::The dosing schedule or timing. Examples: BID, TID, Q12H, daily, PRN, twice daily.",
        "DURATION::The treatment duration. Examples: 7 days, 6 weeks, 3 months, until follow-up.",
    ]) \
    .setFewShotExamples(medication_few_shot)

pipeline_2 = nlp.Pipeline(stages=[document_assembler, sentence_detector, entity_extractor])

In [ ]:
prescription_notes = [
    "DISCHARGE MEDICATIONS: 1. Metformin 1000mg PO BID with meals. 2. Lisinopril 10mg PO daily. "
    "3. Atorvastatin 40mg PO at bedtime. 4. Aspirin 81mg PO daily. 5. Empagliflozin 10mg PO daily. "
    "6. Metoprolol succinate XL 50mg PO daily.",

    "ICU MEDICATION ORDERS: Norepinephrine 0.1 mcg/kg/min IV, titrate to MAP > 65 mmHg. "
    "Vancomycin 1.25g IV Q12H x 14 days. Piperacillin-tazobactam 4.5g IV Q6H x 14 days. "
    "Propofol 5 mcg/kg/min IV continuous for sedation. Pantoprazole 40mg IV daily.",

    "Patient presented to ED with hypertensive urgency, BP 195/115 mmHg. "
    "Administered labetalol 20mg IV push; repeat dose of 40mg IV at 10 minutes. "
    "Discharged on amlodipine 10mg PO daily and hydralazine 25mg PO TID. "
    "Clonidine 0.1mg PO PRN added for BP spikes above 160 systolic.",

    "Post-operative analgesia orders: Morphine 4mg IV Q4H PRN for pain > 7/10. "
    "Acetaminophen 1g PO Q6H scheduled. Ketorolac 30mg IV Q6H x 5 days. "
    "Ondansetron 4mg IV Q8H PRN nausea. Enoxaparin 40mg SQ daily for DVT prophylaxis.",

    "Antibiotic treatment plan for community-acquired pneumonia: "
    "Azithromycin 500mg PO daily x 5 days PLUS amoxicillin-clavulanate 875mg/125mg PO BID x 7 days. "
    "Prednisone 40mg PO daily x 5 days for inflammatory component. "
    "Albuterol inhaler 2 puffs Q4H PRN for bronchospasm.",
]

data_2 = spark.createDataFrame(
    [(f"doc_{i+1}", t) for i, t in enumerate(prescription_notes)]
).toDF("doc_id", "text")
data_2.show(20, truncate=100)

In [ ]:
%%time
result_2 = pipeline_2.fit(data_2).transform(data_2).cache()
result_2.select("doc_id", "entities.result").show(20, truncate=100)

In [ ]:
result_2.selectExpr("doc_id", "explode(entities) AS entity") \
    .select(
        F.col("doc_id"),
        F.col("entity.metadata")["sentence"].cast("int").alias("sentence_id"),
        F.col("entity.result").alias("entity_text"),
        F.col("entity.begin").alias("begin"),
        F.col("entity.end").alias("end"),
        F.col("entity.metadata")["entity"].alias("entity_type"),
    ).show(20, truncate=100)

# 3. Clinical De-identification — PHI Extraction

Protected Health Information (PHI) de-identification is a critical compliance requirement under HIPAA. Before sharing or analyzing clinical notes, PHI must be detected and removed or masked.

`MedicalLLMEntityExtractor` can extract the following PHI categories defined by HIPAA's Safe Harbor standard:

| PHI Type | Examples |
|---|---|
| `PATIENT` | Patient names |
| `DOCTOR` | Physician / provider names |
| `HOSPITAL` | Hospital or clinic names |
| `DATE` | Admission dates, DOB, appointment dates |
| `AGE` | Patient age (especially if > 89) |
| `LOCATION` | Street addresses, cities, states |
| `PHONE` | Phone or fax numbers |
| `ID` | MRN, SSN, insurance IDs, account numbers |

Few-shot examples significantly improve PHI detection accuracy on clinical free text.

In [ ]:
phi_few_shot = [
    (
        "Patient John Davis, DOB 06/12/1960, MRN 482-29-1837, seen at Boston Medical Center.",
        '{"extractions": [{"entity": "PATIENT", "text": "John Davis"}, '
        '{"entity": "DATE", "text": "06/12/1960"}, '
        '{"entity": "ID", "text": "482-29-1837"}, '
        '{"entity": "HOSPITAL", "text": "Boston Medical Center"}]}'
    ),
    (
        "Referred by Dr. Maria Santos, phone 617-555-3842, address 100 Main St, Cambridge MA 02139.",
        '{"extractions": [{"entity": "DOCTOR", "text": "Dr. Maria Santos"}, '
        '{"entity": "PHONE", "text": "617-555-3842"}, '
        '{"entity": "LOCATION", "text": "100 Main St, Cambridge MA 02139"}]}'
    ),
    (
        "92-year-old female Margaret Thompson, SSN 521-44-7893, admitted January 5, 2024.",
        '{"extractions": [{"entity": "AGE", "text": "92-year-old"}, '
        '{"entity": "PATIENT", "text": "Margaret Thompson"}, '
        '{"entity": "ID", "text": "521-44-7893"}, '
        '{"entity": "DATE", "text": "January 5, 2024"}]}'
    ),
]

entity_extractor \
    .setEntityTypes([
        "PATIENT::The full name of the patient as written in the text.",
        "DOCTOR::The name of the physician, provider, or clinical staff member.",
        "HOSPITAL::The name of a hospital, clinic, medical center, or healthcare facility.",
        "DATE::Any date including admission dates, discharge dates, dates of birth, or appointment dates.",
        "AGE::The patient's age or age range, especially if 90 years or older.",
        "LOCATION::A street address, city, state, zip code, or geographic location.",
        "PHONE::A telephone number, fax number, or pager number.",
        "ID::A patient identifier such as MRN, SSN, insurance ID, account number, or episode number.",
    ]) \
    .setFewShotExamples(phi_few_shot)

pipeline_3 = nlp.Pipeline(stages=[document_assembler, sentence_detector, entity_extractor])

In [ ]:
clinical_notes_phi = [
    "Patient: Sarah Elizabeth Thompson, DOB: 03/14/1968 (Age 56). MRN: 7823-4519. "
    "Attending: Dr. Michael Chen, MD. Admission Date: 01/22/2024. "
    "Facility: Massachusetts General Hospital, 55 Fruit Street, Boston, MA 02114. "
    "Contact: (617) 555-0847. Insurance: Aetna #AET-8823-01.",

    "Referred by Dr. Amanda Rodriguez from Cleveland Clinic. "
    "Patient: James Patrick O'Brien, SSN: 521-44-7893, DOB: November 8, 1955. "
    "Address: 2847 Oak Lane, Chicago, IL 60614. Home: 312-555-9021. "
    "Emergency contact: Mary O'Brien (wife) at 312-555-7634. "
    "Follow-up appointment: February 15, 2024 at 2:00 PM.",

    "Post-operative note. Patient: Michael David Ramirez, Age 43. "
    "Admitted to Johns Hopkins Hospital on December 3, 2023. "
    "Surgeon: Dr. Patricia Kim, MD, FACS. Assistant surgeon: Dr. Kevin Park. "
    "Patient discharged December 7, 2023. Fax: (410) 955-2900. "
    "Account #: 992847-2024. Follow-up with Dr. Kim on January 10, 2024.",

    "Consultation note from Dr. Lisa Wang, Cardiology, Cedars-Sinai Medical Center. "
    "Patient: Helen Marie Johnson, 78 years old. MRN: HC-4829-221. "
    "Primary care provider: Dr. Robert Garcia, phone (310) 855-3311. "
    "Lives at 5502 Wilshire Blvd Apt 14B, Los Angeles, CA 90036. "
    "Consultation requested on 03/08/2024. Insurance: Medicare #1EG4-TE5-MK72.",
]

data_3 = spark.createDataFrame(
    [(f"doc_{i+1}", t) for i, t in enumerate(clinical_notes_phi)]
).toDF("doc_id", "text")
data_3.show(20, truncate=100)

In [ ]:
%%time
result_3 = pipeline_3.fit(data_3).transform(data_3).cache()
result_3.select("doc_id", "entities.result").show(20, truncate=100)

In [ ]:
result_3.selectExpr("doc_id", "explode(entities) AS entity") \
    .select(
        F.col("doc_id"),
        F.col("entity.metadata")["sentence"].cast("int").alias("sentence_id"),
        F.col("entity.result").alias("phi_text"),
        F.col("entity.begin").alias("begin"),
        F.col("entity.end").alias("end"),
        F.col("entity.metadata")["entity"].alias("phi_type"),
    ).show(20, truncate=100)

# 4. Oncology Named Entity Recognition with Few-Shot Examples

Oncology notes contain highly specialized terminology — cancer types, staging systems, biomarkers, and targeted therapies. Few-shot examples are especially valuable here to guide the LLM on the precise granularity and format of oncology entities.

This example extracts six oncology-specific entity types:

| Entity | Description | Example |
|---|---|---|
| `CANCER_TYPE` | The cancer diagnosis with histological subtype | `NSCLC adenocarcinoma`, `triple-negative breast cancer` |
| `STAGE` | The clinical or pathological staging | `Stage IIIB`, `cT3N2M0`, `ISS Stage III` |
| `BIOMARKER` | Molecular markers and genomic alterations | `EGFR exon 19 deletion`, `PD-L1 TPS 45%`, `HER2-` |
| `TREATMENT` | Cancer-directed therapies | `osimertinib 80mg`, `FOLFOX`, `pembrolizumab` |
| `SIDE_EFFECT` | Adverse treatment effects | `grade 2 neutropenia`, `acneiform rash` |
| `RESPONSE` | Treatment response assessment | `partial response`, `complete remission`, `progressive disease` |

In [ ]:
oncology_few_shot = [
    (
        "Patient has stage IV melanoma with BRAF V600E mutation, started on vemurafenib 960mg BID. "
        "Developed grade 3 arthralgias requiring dose reduction.",
        '{"extractions": [{"entity": "CANCER_TYPE", "text": "stage IV melanoma"}, '
        '{"entity": "STAGE", "text": "stage IV"}, '
        '{"entity": "BIOMARKER", "text": "BRAF V600E mutation"}, '
        '{"entity": "TREATMENT", "text": "vemurafenib 960mg BID"}, '
        '{"entity": "SIDE_EFFECT", "text": "grade 3 arthralgias"}]}'
    ),
    (
        "ER+/PR+/HER2- breast cancer stage IIB (T2N1M0). Treated with anastrozole 1mg daily. "
        "CT shows complete response after 3 cycles.",
        '{"extractions": [{"entity": "CANCER_TYPE", "text": "ER+/PR+/HER2- breast cancer"}, '
        '{"entity": "STAGE", "text": "stage IIB (T2N1M0)"}, '
        '{"entity": "BIOMARKER", "text": "ER+"}, '
        '{"entity": "BIOMARKER", "text": "PR+"}, '
        '{"entity": "BIOMARKER", "text": "HER2-"}, '
        '{"entity": "TREATMENT", "text": "anastrozole 1mg daily"}, '
        '{"entity": "RESPONSE", "text": "complete response"}]}'
    ),
    (
        "Colorectal adenocarcinoma Stage IV (T4N2M1) with liver metastases. KRAS wild-type, MSI-H. "
        "Initiated FOLFOX plus bevacizumab. Grade 2 peripheral neuropathy noted.",
        '{"extractions": [{"entity": "CANCER_TYPE", "text": "Colorectal adenocarcinoma"}, '
        '{"entity": "STAGE", "text": "Stage IV (T4N2M1)"}, '
        '{"entity": "BIOMARKER", "text": "KRAS wild-type"}, '
        '{"entity": "BIOMARKER", "text": "MSI-H"}, '
        '{"entity": "TREATMENT", "text": "FOLFOX"}, '
        '{"entity": "TREATMENT", "text": "bevacizumab"}, '
        '{"entity": "SIDE_EFFECT", "text": "Grade 2 peripheral neuropathy"}]}'
    ),
]

entity_extractor \
    .setEntityTypes([
        "CANCER_TYPE::The cancer diagnosis with histological subtype. Examples: NSCLC adenocarcinoma, triple-negative breast cancer, multiple myeloma.",
        "STAGE::The clinical or pathological stage. Examples: Stage IIIB, cT3N2M0, Stage IV, ISS Stage II.",
        "BIOMARKER::A molecular marker, genomic alteration, or receptor status. Examples: EGFR exon 19 deletion, PD-L1 TPS 45%, HER2-, KRAS G12C, ALK rearrangement.",
        "TREATMENT::A cancer-directed therapy including chemotherapy, immunotherapy, targeted therapy, or radiation. Examples: osimertinib 80mg PO daily, pembrolizumab 200mg Q3W, FOLFOX.",
        "SIDE_EFFECT::An adverse effect or toxicity from cancer treatment. Examples: grade 2 neutropenia, acneiform rash, peripheral neuropathy.",
        "RESPONSE::The treatment response assessment. Examples: partial response, complete remission, progressive disease, stable disease.",
    ]) \
    .setFewShotExamples(oncology_few_shot)

pipeline_4 = nlp.Pipeline(stages=[document_assembler, sentence_detector, entity_extractor])

In [ ]:
oncology_notes = [
    "62-year-old male with newly diagnosed non-small cell lung cancer (NSCLC), adenocarcinoma subtype, "
    "clinical stage IIIB (cT3N2M0). Molecular profiling: EGFR exon 19 deletion positive, "
    "ALK rearrangement negative, PD-L1 TPS 45%. Initiated osimertinib 80mg PO daily as first-line "
    "targeted therapy. Post-cycle 1 CT chest: partial response with 35% reduction in primary lesion.",

    "54-year-old female with metastatic breast cancer (ER+/PR+/HER2-) with bone and liver metastases, "
    "clinical stage IV. Previously treated with tamoxifen then letrozole (both with progression). "
    "CDK4/6 inhibitor ribociclib 600mg PO daily (21/7 schedule) added to fulvestrant 500mg IM monthly. "
    "Grade 2 neutropenia observed at cycle 2; ribociclib dose reduced to 400mg. "
    "Restaging CT at cycle 4 demonstrates stable disease.",

    "72-year-old male with multiple myeloma, ISS Stage III (high-risk cytogenetics: del 17p). "
    "Completed 6 cycles of VRd (bortezomib 1.3mg/m2 SQ, lenalidomide 25mg PO days 1-14, "
    "dexamethasone 40mg weekly). Achieved very good partial response (VGPR). "
    "ASCT performed 8 months ago. Now relapsed: serum M-protein 3.2 g/dL, new lytic lesions. "
    "Initiating daratumumab, pomalidomide, and dexamethasone (DPd) protocol.",

    "48-year-old female with pancreatic ductal adenocarcinoma, Stage IIB (T3N1M0). "
    "BRCA2 germline mutation confirmed. Resection performed (R0 margin). "
    "Adjuvant modified FOLFIRINOX initiated: oxaliplatin 85mg/m2, leucovorin 400mg/m2, "
    "irinotecan 150mg/m2, fluorouracil 2400mg/m2 over 46 hours, Q2W x 12 cycles. "
    "Grade 3 diarrhea and grade 2 fatigue required one dose delay at cycle 5. "
    "Post-adjuvant imaging: no evidence of disease (NED) — complete response.",
]

data_4 = spark.createDataFrame(
    [(f"doc_{i+1}", t) for i, t in enumerate(oncology_notes)]
).toDF("doc_id", "text")
data_4.show(20, truncate=100)

In [ ]:
%%time
result_4 = pipeline_4.fit(data_4).transform(data_4).cache()
result_4.select("doc_id", "entities.result").show(20, truncate=100)

In [ ]:
result_4.selectExpr("doc_id", "explode(entities) AS entity") \
    .select(
        F.col("doc_id"),
        F.col("entity.metadata")["sentence"].cast("int").alias("sentence_id"),
        F.col("entity.result").alias("entity_text"),
        F.col("entity.begin").alias("begin"),
        F.col("entity.end").alias("end"),
        F.col("entity.metadata")["entity"].alias("entity_type"),
    ).show(20, truncate=100)

# 5. Custom Prompt Template — Adverse Drug Reaction Extraction

By default, `MedicalLLMEntityExtractor` uses a built-in prompt template. You can replace it entirely with `setPromptTemplate()` for specialized extraction tasks.

The custom prompt template must include these two placeholders:
- **`{entityTypes}`** — expanded list of entity type names (and descriptions if provided)
- **`{text}`** — the input clinical text

An optional **`{examples}`** placeholder includes the few-shot examples block when provided.

This example demonstrates **Adverse Drug Reaction (ADR) extraction** from patient-reported medication reviews, using a domain-specific prompt that emphasizes the ADR extraction task.

In [ ]:
adr_prompt_template = (
    "You are a pharmacovigilance expert specializing in adverse drug reaction detection. "
    "Your task is to extract drug-related entities from patient medication reports.\n\n"
    "Extract the following entity types:\n{entityTypes}\n\n"
    "Rules:\n"
    "- Extract EXACT text spans as they appear in the report\n"
    "- A single drug may have multiple adverse reactions — extract each separately\n"
    "- Include severity qualifiers (e.g., 'severe', 'mild') as part of the ADVERSE_REACTION text\n"
    "- Return ONLY valid JSON in the specified format\n\n"
    "{examples}"
    "Patient medication report:\n{text}\n\n"
    "JSON extraction:"
)

adr_few_shot = [
    (
        "I've been taking Lipitor 20mg for 6 months and developed severe muscle cramps and weakness in my legs.",
        '{"extractions": [{"entity": "DRUG", "text": "Lipitor"}, '
        '{"entity": "DOSAGE", "text": "20mg"}, '
        '{"entity": "ADVERSE_REACTION", "text": "severe muscle cramps"}, '
        '{"entity": "ADVERSE_REACTION", "text": "weakness in my legs"}]}'
    ),
    (
        "Metformin 500mg twice a day causes me terrible nausea and diarrhea, especially in the morning.",
        '{"extractions": [{"entity": "DRUG", "text": "Metformin"}, '
        '{"entity": "DOSAGE", "text": "500mg"}, '
        '{"entity": "FREQUENCY", "text": "twice a day"}, '
        '{"entity": "ADVERSE_REACTION", "text": "terrible nausea"}, '
        '{"entity": "ADVERSE_REACTION", "text": "diarrhea"}]}'
    ),
]

entity_extractor \
    .setEntityTypes([
        "DRUG::The medication or drug name mentioned in the report.",
        "DOSAGE::The dose amount and unit of the medication.",
        "FREQUENCY::How often the medication is taken.",
        "ADVERSE_REACTION::A side effect, adverse event, or unwanted reaction to the medication. Include severity descriptors.",
    ]) \
    .setFewShotExamples(adr_few_shot) \
    .setPromptTemplate(adr_prompt_template)

pipeline_5 = nlp.Pipeline(stages=[document_assembler, sentence_detector, entity_extractor])

In [ ]:
patient_drug_reports = [
    "I started taking Lisinopril 10mg daily for blood pressure about 3 weeks ago. "
    "Within a few days I developed a persistent, dry, hacking cough that hasn't gone away. "
    "Also noticed some mild dizziness when I stand up quickly.",

    "Been on Sertraline 50mg every morning for depression for 2 months. "
    "The first 2 weeks were rough — extreme nausea, headaches, and trouble sleeping. "
    "Now I have decreased libido and I feel emotionally blunted most of the time.",

    "My doctor prescribed Prednisone 40mg daily for my asthma flare. "
    "After 5 days I have terrible insomnia, my blood sugar is through the roof, "
    "and I'm incredibly irritable and anxious. Also gaining weight rapidly.",

    "Taking Warfarin 5mg daily for my AFib. I bruise incredibly easily now — "
    "even minor bumps leave huge bruises. Had a nosebleed last week that lasted 20 minutes. "
    "My last INR was 3.8, which my doctor said was too high.",

    "Amiodarone 200mg twice a day for my heart arrhythmia. "
    "Noticed my skin becomes extremely sensitive to sunlight — bad sunburn after just 10 minutes outside. "
    "Also feel short of breath when climbing stairs, which is new. "
    "My thyroid tests came back abnormal at last check.",
]

data_5 = spark.createDataFrame(
    [(f"doc_{i+1}", t) for i, t in enumerate(patient_drug_reports)]
).toDF("doc_id", "text")
data_5.show(20, truncate=100)

In [ ]:
%%time
result_5 = pipeline_5.fit(data_5).transform(data_5).cache()
result_5.select("doc_id", "entities.result").show(20, truncate=100)

In [ ]:
result_5.selectExpr("doc_id", "explode(entities) AS entity") \
    .select(
        F.col("doc_id"),
        F.col("entity.metadata")["sentence"].cast("int").alias("sentence_id"),
        F.col("entity.result").alias("entity_text"),
        F.col("entity.begin").alias("begin"),
        F.col("entity.end").alias("end"),
        F.col("entity.metadata")["entity"].alias("entity_type"),
    ).show(20, truncate=100)

# Summary

This notebook demonstrated five clinical use cases for `MedicalLLMEntityExtractor`. All pipelines share a single `SentenceDetectorDLModel` + `MedicalLLMEntityExtractor` pair loaded once at the start, with entity types reconfigured between sections.

| # | Use Case | Key Entity Types | Key Feature Shown |
|---|---|---|---|
| 1 | Default Clinical NER | PROBLEM, TEST, TREATMENT | Default entity types with descriptions (`::` syntax) |
| 2 | Medication Extraction | DRUG, DOSAGE, ROUTE, FREQUENCY, DURATION | Few-shot examples for clinical abbreviations |
| 3 | De-identification (PHI) | PATIENT, DOCTOR, HOSPITAL, DATE, AGE, LOCATION, PHONE, ID | Custom entity types for HIPAA compliance |
| 4 | Oncology NER | CANCER_TYPE, STAGE, BIOMARKER, TREATMENT, SIDE_EFFECT, RESPONSE | Domain-specific few-shot with complex terminology |
| 5 | Adverse Drug Reactions | DRUG, DOSAGE, FREQUENCY, ADVERSE_REACTION | Custom prompt template for specialized tasks |

### Shared Pipeline Architecture

```
nlp.DocumentAssembler → nlp.SentenceDetectorDLModel → medical.MedicalLLMEntityExtractor
         ↓                          ↓                               ↓
     DOCUMENT                SENTENCE (per sentence)          CHUNK (entities)
```

### Key API Methods

| Method | Purpose |
|---|---|
| `.setEntityTypes([...])` | Define entity types; use `::` for per-type descriptions |
| `.setFewShotExamples([(text, json), ...])` | Provide examples to guide extraction |
| `.setPromptTemplate(template)` | Override the default prompt; use `{entityTypes}`, `{text}`, `{examples}` |
| `.setNPredict(n)` | Max tokens to generate (increase for many entities per document) |
| `.setTemperature(t)` | Sampling temperature (0.0–0.2 recommended for deterministic extraction) |
| `.setNGpuLayers(n)` | GPU layers to offload (set to 99 to use full GPU) |
| `.setNCtx(n)` | Context window size (increase for longer documents) |
| `.setBatchSize(n)` | Documents processed per batch (increase for throughput) |

### Related Annotators
- **`medical.MedicalLLM`** — general-purpose LLM inference (summarization, Q&A, RAG, chat)
- **`medical.MedicalNerModel`** — fast BiLSTM-CRF NER for fixed, high-throughput entity schemas
- **`medical.NerConverterInternal`** — convert `NAMED_ENTITY` outputs to `CHUNK` for downstream tasks